# 统一标签输入与灰客户

`target` 与 `overdue+dpds` 是两种互斥模式。target 可为列名、一维数据、callable 或标签字典。
非 0/1 值在 del_grey=True 时逐标签剔除，否则按负样本 0 计算。

In [ ]:
import numpy as np
import pandas as pd
import hscredit
from hscredit.report import feature_bin_stats, ModelReport
from hscredit.core.rules import Rule
from hscredit.core.eda.target import bad_rate_overall

data = pd.DataFrame({
    "评分": np.arange(12, dtype=float),
    "MOB1": [0, 8, 2, 3, 0, 8, 0, 8, 3, 1, 0, 8],
    "目标列": [0, 1, 2, -1, 0, 1, 0, 1, np.nan, 2, 0, 1],
    "金额": np.arange(1, 13) * 100,
})
data


## 单标签的多种来源

Series 按索引对齐，数组和列表按行位置对应；callable 接收当前完整作用数据。

In [ ]:
inputs = ["目标列", data["目标列"], data["目标列"].to_numpy(),
          data["目标列"].tolist(), lambda df: df["目标列"]]
for target in inputs:
    result = bad_rate_overall(data, target=target, del_grey=True)
    assert result["样本总数"] == 8 and result["坏样本数"] == 4
kept = bad_rate_overall(data, target="目标列", del_grey=False)
assert kept["样本总数"] == 12 and kept["坏样本数"] == 4
kept


## 自定义多标签和金额口径

字典键原样显示。每个标签有自己的有效样本和金额分母。

In [ ]:
targets = {
    "自定义风险_7+": lambda df: df["目标列"],
    "任意逾期": lambda df: df["MOB1"] > 0,
}
table = feature_bin_stats(data, "评分", target=targets, del_grey=True,
                          rules=[3.5, 7.5], margins=True, n_jobs=1)
assert table.iloc[-1][("自定义风险_7+", "样本总数")] == 8
assert table.iloc[-1][("任意逾期", "样本总数")] == 12
amount_table = feature_bin_stats(data, "评分", target=targets, del_grey=True,
                                 rules=[3.5, 7.5], amount="金额", margins=True, n_jobs=1)
expected_amount = data.loc[data["目标列"].isin([0, 1]), "金额"].sum()
assert amount_table.iloc[-1][("自定义风险_7+", "样本总数")] == expected_amount
table


## 逾期模式独立传参

默认 > 的灰区为 (0, dpds]；>= 的灰区为 (0, dpds)。< 和 <= 暂无灰区。

In [ ]:
overdue_table = feature_bin_stats(data, "评分", overdue="MOB1", dpds=[7, 3, 0],
                                  overdue_operator=">", del_grey=True,
                                  rules=[3.5, 7.5], margins=True, n_jobs=1)
for threshold in [7, 3, 0]:
    valid = ~data["MOB1"].between(0, threshold, inclusive="right")
    assert overdue_table.iloc[-1][(f"MOB1_{threshold}+", "样本总数")] == valid.sum()
overdue_table


## 规则与多数据集模型报告

同一个 callable 在每个数据集分别执行一次。规则过滤和报告取数复用已生成标签。

In [ ]:
rule_table = Rule("评分 >= 4", n_jobs=1).report(data, target=targets, del_grey=True, margins=True)
from sklearn.dummy import DummyClassifier
model = DummyClassifier(strategy="prior").fit(data[["评分"]], data["目标列"].eq(1))
report = ModelReport(model, datasets={"训练集": data, "测试集": data.iloc[:8]},
                     features=["评分"], target=targets, del_grey=True, n_jobs=1)
summary = report.summary()
assert summary.loc["自定义风险_7+", ("样本数", "训练集")] == 8
assert summary.loc["自定义风险_7+", ("样本数", "测试集")] == 6
summary


## 四种标签方向与阈值边界

逐行独立核算样本数和正样本数，验证 DPD 边界及开启、关闭剔灰后的结果。


In [ ]:
boundary = pd.DataFrame({"逾期天数": [-1, 0, 1, 3, 4, np.nan]})
days = boundary["逾期天数"]
comparisons = {">": days > 3, ">=": days >= 3, "<": days < 3, "<=": days <= 3}
for operator, positive in comparisons.items():
    for grey in (False, True):
        valid = pd.Series(True, index=boundary.index)
        if grey and operator == ">":
            valid = ~((days > 0) & (days <= 3))
        elif grey and operator == ">=":
            valid = ~((days > 0) & (days < 3))
        actual = bad_rate_overall(boundary, overdue="逾期天数", dpds=3,
                                  overdue_operator=operator, del_grey=grey).iloc[0]
        assert actual["样本总数"] == valid.sum()
        assert actual["坏样本数"] == (valid & positive).sum()


## 自动参考集与重复业务索引

置换分析先在原始数据上计算标签，再归一化重复索引。未指定参考集时复用标签，callable 不会重复执行。


In [ ]:
from hscredit.report import rule_swap_analysis

repeated = data.set_axis(np.repeat(np.arange(6), 2))
calls = []
def swap_target(frame):
    calls.append(frame.index.copy())
    return frame["目标列"]

swap = rule_swap_analysis(repeated, score="评分", target=swap_target,
                          del_grey=True, rules_out=[Rule("评分 >= 4")],
                          method="quantile", max_n_bins=3, n_jobs=1)
assert len(calls) == 1 and calls[0].equals(repeated.index)
assert swap["swap_pipeline"].iloc[0]["样本总数"] == 8
swap["swap_pipeline"]


## 真实放款数据验证

使用本地 `hscredit_yyp.xlsx`，对 MOB1 的 7、3、0 天阈值检查四种方向；显式 FPD 标签验证金额分母。


In [ ]:
from pathlib import Path

workbook = next((p for p in [Path("hscredit_yyp.xlsx"), Path("examples/hscredit_yyp.xlsx")]
                 if p.is_file()), None)
if workbook is None:
    raise FileNotFoundError("请在项目根目录或 examples 目录运行，并提供 hscredit_yyp.xlsx")
loans = pd.read_excel(workbook)
for operator in [">", ">=", "<", "<="]:
    actual = bad_rate_overall(loans, overdue=["MOB1"], dpds=[7, 3, 0],
                              overdue_operator=operator, del_grey=True)
    for threshold, (_, row) in zip([7, 3, 0], actual.iterrows()):
        days = loans["MOB1"]
        positive = {">": days > threshold, ">=": days >= threshold,
                    "<": days < threshold, "<=": days <= threshold}[operator]
        valid = pd.Series(True, index=loans.index)
        if operator == ">":
            valid = ~((days > 0) & (days <= threshold))
        elif operator == ">=":
            valid = ~((days > 0) & (days < threshold))
        assert row["样本总数"] == valid.sum()
        assert row["坏样本数"] == (valid & positive).sum()

loan_table = feature_bin_stats(loans, "衡枢鉴真分老客版", target=lambda df: df["FPD"],
                               amount="放款金额", del_grey=True, method="quantile",
                               margins=True, n_jobs=1)
assert loan_table.iloc[-1]["样本总数"] == loans.loc[loans.FPD.isin([0, 1]), "放款金额"].sum()
loan_table
